# Conclusiones de mineria de datos

In [501]:
import pandas as pd

#### - ¿Donde se aconseja realizar un explotacion agrícola para obtener un beneficio entre 5 % y 10 %?

Para el calculo de los beneficios se necesitan los ingresos (producción por precio de venta tonelada) y los costes, para los costes necesitamos el coste del fertilizante, vamos a colocar una estimación de 1.20 USD por kg, el coste del agua que vamoas a estimar una coste de agua de 0.05 usd por m3 y vamos a asumir unos costes fijos base (mano de obra, maquinaria, semillas) de unos 300 usd.

In [502]:
df_precio = pd.read_csv('datos/precios_mercado_limpios.csv')
df_agricola = pd.read_csv('datos/produccion_agricola.csv')

In [503]:
df_agricola.head()


,pais,codigo_iso,region,cultivo,anio,superficie_hectareas,rendimiento_ton_ha,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,tendencia_5_anios
0,Argentina,ARG,América del Sur,Soja,2011,4703643,2.22,10465461,105,4811,0.000536
1,Argentina,ARG,América del Sur,Maíz,2011,3371781,8.45,28485921,164,5557,0.030943
2,Argentina,ARG,América del Sur,Cebada,2011,1661577,2.77,4596917,105,6514,-0.039779
3,Argentina,ARG,América del Sur,Té,2011,188518,3.07,577886,236,8527,0.003623
4,Argentina,ARG,América del Sur,Algodón,2011,1638327,1.18,1929756,276,6925,0.007735


Filtro de años (Los años 2019/2020 del dataset de producción agricola se trataran como si fueran los años 2021/2022 para poder trabajar con el dataset de precios de mercado)

In [504]:
df_agricola_reciente = df_agricola[df_agricola['anio'].isin([2019, 2020])].copy()
df_agricola_reciente['anio'] = df_agricola_reciente['anio'].replace({2019: 2021, 2020: 2022})
df_agricola_reciente.head()

,pais,codigo_iso,region,cultivo,anio,superficie_hectareas,rendimiento_ton_ha,produccion_ton,fertilizantes_kg_ha,agua_riego_m3_ha,tendencia_5_anios
64,Argentina,ARG,América del Sur,Soja,2021,1321690,3.14,4151744,188,8633,-0.008423
65,Argentina,ARG,América del Sur,Maíz,2021,6550902,7.19,47122614,55,7232,-0.002302
66,Argentina,ARG,América del Sur,Cebada,2021,803243,6.66,5350442,157,9818,0.024752
67,Argentina,ARG,América del Sur,Té,2021,123988,1.79,221839,149,6507,0.016887
68,Argentina,ARG,América del Sur,Algodón,2021,832592,4.71,3917824,186,1441,0.040693


Preparacion de los precios

In [505]:
df_precios_medios = df_precio.groupby(['pais', 'producto'])['precio_usd_ton'].mean().reset_index()
df_precios_medios.rename(columns={'producto': 'cultivo'}, inplace=True)

Preparación de la produccion (media historica por pais y cultivo)

In [506]:
df_ag_media = df_agricola_reciente.groupby(['pais', 'cultivo']).agg({
    'rendimiento_ton_ha': 'mean',
    'fertilizantes_kg_ha': 'mean',
    'agua_riego_m3_ha': 'mean'
}).reset_index()

In [507]:
df_rentabilidad = pd.merge(df_ag_media, df_precios_medios, on=['pais', 'cultivo'], how='inner')

In [508]:
df_rentabilidad.head()

,pais,cultivo,rendimiento_ton_ha,fertilizantes_kg_ha,agua_riego_m3_ha,precio_usd_ton
0,Alemania,Maíz,8.485,160.0,8698.5,202.710000
1,Alemania,Soja,3.850,148.5,3237.5,513.152308
2,Alemania,Trigo,6.920,130.5,3252.5,314.085000
3,Argentina,Maíz,6.860,150.5,7137.5,201.473333
4,Argentina,Soja,2.840,198.5,6064.5,502.715714


Variables de costes

In [509]:
PRECIO_FERTILIZANTE_KG = 2
PRECIO_AGUA_M3 = 0.01
COSTE_BASE_HA = 650
COSTE_LOGISTICO_TON = 80.0


Calculo de ingresos y costes

In [510]:
# Ingresos
df_rentabilidad['ingresos_por_ha_usd'] = df_rentabilidad['rendimiento_ton_ha'] * df_rentabilidad['precio_usd_ton']

In [511]:
# Costes
df_rentabilidad['costes_por_ha_usd'] = (
    COSTE_BASE_HA + 
    (df_rentabilidad['fertilizantes_kg_ha'] * PRECIO_FERTILIZANTE_KG) + 
    (df_rentabilidad['agua_riego_m3_ha'] * PRECIO_AGUA_M3) +
    (df_rentabilidad['rendimiento_ton_ha'] * COSTE_LOGISTICO_TON)
)

Beneficio y margen

In [512]:
# Beneficio y Margen
df_rentabilidad['beneficio_por_ha_usd'] = df_rentabilidad['ingresos_por_ha_usd'] - df_rentabilidad['costes_por_ha_usd']
df_rentabilidad['margen_beneficio_pct'] = (df_rentabilidad['beneficio_por_ha_usd'] / df_rentabilidad['ingresos_por_ha_usd']) * 100

Filtrar entre 5% y 10%

In [513]:
df_pais_rentabilidad = df_rentabilidad.groupby('pais').agg({
    'margen_beneficio_pct': 'mean'
}).reset_index()

paises_ideales = df_pais_rentabilidad[
    (df_pais_rentabilidad['margen_beneficio_pct'] >= 5) & 
    (df_pais_rentabilidad['margen_beneficio_pct'] <= 10)
].sort_values(by='margen_beneficio_pct', ascending=False)

print("--- UBICACIONES MACROECONÓMICAS RECOMENDADAS (5% - 10%) ---")
print(paises_ideales)

--- UBICACIONES MACROECONÓMICAS RECOMENDADAS (5% - 10%) ---
        pais  margen_beneficio_pct
9      Kenia              8.614267
5     España              7.632647
2  Australia              6.369645
1  Argentina              6.343754


Para determinar las ubicaciones geográficas más viables y sostenibles para el desarrollo del proyecto, se ha llevado a cabo un análisis de rentabilidad unitario por hectárea. Asumimos una proyección de los datos productivos de los dos últimos años registrados (2019 y 2020) para equipararlos con la ventana temporal de los precios de mercado disponibles (2021 y 2022). Esto permite calcular ingresos realistas basados en rendimientos recientes y precios actuales.
Para el caculo de los beneficios es necesario saber cuales han sido los costes fijos y variables de la producción. Los parámetros que hemos establecido han sido:
- Coste Fijo Base: 650 USD/ha (cubre gastos estructurales como maquinaria, mano de obra fija y mantenimiento de las tierras).
- Precio del Fertilizante: 2 USD/kg.
- Precio del Agua de Riego: 0.01 USD/m³.
- Coste Logístico: 80 USD/tonelada (asociado al transporte y manejo del volumen final de cosecha).
A partir de estos valores, se calcularon los costes totales y los ingresos (Rendimiento × Precio de venta) para obtener el margen de beneficio porcentual medio de cada país.

El criterio de selección final se ha centrado en identificar aquellos países que presentan un margen de beneficio medio situado entre el 5% y el 10%. Este rango se considera ya que márgenes inferiores al 5% suponen un riesgo financiero y márgenes superiores al 10% a nivel macroeconómico suelen representar nichos de mercado altamente subsidiados o anomalías temporales.

Resultados----

#### - ¿Donde se aconseja realizar un explotación ganadera para obtener el mismo beneficio anterior?

Para este punto vamos a tratar la producción ganadera solo con la carne, ya que el dataset de precios de mercado no diferencia la leche de que tipo de ganado es.

In [514]:
df_ganadera = pd.read_csv('datos/produccion_ganadera.csv')
df_ganadera_reciente = df_ganadera[df_ganadera['anio'].isin([2019, 2020])].copy()
df_ganadera_reciente['anio'] = df_ganadera_reciente['anio'].replace({2019: 2021, 2020: 2022})
df_ganadera_reciente.head()

,pais,anio,tipo_ganado,cabezas_ganado,produccion_carne_ton,produccion_leche_lt,emisiones_ch4_ton_co2eq,intensidad_emisiones,eficiencia_carne
40,Argentina,2021,bovino,7593050,4326749,6553035977,9460926,1.246,0.5698
41,Argentina,2021,porcino,5086842,918331,0,14237642,2.799,0.1805
42,Argentina,2021,avicola,388626399,1284364,0,17202956,0.044,0.0033
43,Argentina,2021,ovino,12696608,291906,388028119,5186803,0.409,0.0230
44,Argentina,2021,caprino,9474573,234999,256544950,10633115,1.122,0.0248


Convertir tipo de ganado a "carne_tipo", para poder combinar con dataset de precios de mercado

In [515]:
diccionario_carnes = {
    'bovino': 'Carne_bovina',
    'porcino': 'Carne_porcina'
}

In [516]:
df_ganadera_reciente['producto'] = df_ganadera_reciente['tipo_ganado'].str.lower().map(diccionario_carnes)
df_ganadera_reciente = df_ganadera_reciente.dropna(subset=['producto'])

Agrupación de valores y cruce de datos

In [517]:
df_precios_medios_gan = df_precio.groupby(['pais', 'producto'])['precio_usd_ton'].mean().reset_index()

df_gan_media = df_ganadera_reciente.groupby(['pais', 'producto']).agg({
    'produccion_carne_ton': 'mean'
}).reset_index()



In [518]:
df_precios_medios_gan.head()

,pais,producto,precio_usd_ton
0,Alemania,Arroz,361.637500
1,Alemania,Carne_bovina,4103.422000
2,Alemania,Carne_porcina,2460.324167
3,Alemania,Leche,348.636667
4,Alemania,Maíz,202.710000


In [519]:
df_gan_media.head()

,pais,producto,produccion_carne_ton
0,Alemania,Carne_bovina,1486593.0
1,Alemania,Carne_porcina,2727357.0
2,Argentina,Carne_bovina,3976808.0
3,Argentina,Carne_porcina,2078503.5
4,Australia,Carne_bovina,1888644.0


In [520]:
df_rent_ganadera = pd.merge(df_gan_media, df_precios_medios_gan, on=['pais', 'producto'], how='inner')
df_rent_ganadera.head()

,pais,producto,produccion_carne_ton,precio_usd_ton
0,Alemania,Carne_bovina,1486593.0,4103.422000
1,Alemania,Carne_porcina,2727357.0,2460.324167
2,Argentina,Carne_bovina,3976808.0,4035.419167
3,Argentina,Carne_porcina,2078503.5,2453.967273
4,Australia,Carne_bovina,1888644.0,4169.215000


Costes y beneficios

In [521]:
df_rent_ganadera['ingreso_por_ton_usd'] = df_rent_ganadera['precio_usd_ton']

COSTE_ALIMENTACION_TON = 2250   
COSTE_VETERINARIO_TON = 380     
COSTE_AGUA_ENERGIA_TON = 175    
COSTE_LOGISTICO_GAN_TON = 155       

df_rent_ganadera['coste_por_ton_usd'] = (
    COSTE_ALIMENTACION_TON + 
    COSTE_VETERINARIO_TON + 
    COSTE_AGUA_ENERGIA_TON + 
    COSTE_LOGISTICO_GAN_TON
)

df_rent_ganadera['beneficio_por_ton_usd'] = df_rent_ganadera['ingreso_por_ton_usd'] - df_rent_ganadera['coste_por_ton_usd']
df_rent_ganadera['margen_beneficio_pct'] = (df_rent_ganadera['beneficio_por_ton_usd'] / df_rent_ganadera['ingreso_por_ton_usd']) * 100

Resultados

In [522]:
df_pais_rent_ganadera = df_rent_ganadera.groupby('pais').agg({'margen_beneficio_pct': 'mean'}).reset_index()

paises_ideales_ganaderia = df_pais_rent_ganadera[
    (df_pais_rent_ganadera['margen_beneficio_pct'] >= 5) & 
    (df_pais_rent_ganadera['margen_beneficio_pct'] <= 10)
].sort_values(by='margen_beneficio_pct', ascending=False)

print("\n--- UBICACIONES GANADERAS RECOMENDADAS (Margen 5% - 10%) ---")
print(paises_ideales_ganaderia)


--- UBICACIONES GANADERAS RECOMENDADAS (Margen 5% - 10%) ---
              pais  margen_beneficio_pct
4            China              6.789541
6   Estados Unidos              6.689828
3           Brasil              6.624244
10          México              6.573656
8            India              6.076890
9            Kenia              5.720576
5           España              5.570526


Para determinar que lugares tiene un beneficio situado entre el 5% y el 10% de con la producción agricola, hemos alineado los precios de los años 2021/2022 con los años de producción de 2019/2020. El análisis se ha centrado tan solo en la carne bovina y porcina puesto que son los únicos datos de precio que tenemos en el dataset de precios de mercado. 

Para calcular el beneficio ha sido necesario establecer unos costes, para ello hemos establecido unos costes promedios de la indutrica cárnica global. Los costes han sido los siguientes:
- Alimentación y forrajes: 2.250 USD / ton.
- Sanidad y cuidados veterinarios: 380 USD / ton.
- Agua, energía y mantenimiento: 175 USD / ton.
- Logística y cadena de frío: 155 USD / ton.

Hemos empleado un análisis unitario para el calculo de los ingresos, para evitar que predominen paises más grandes y que esto haga que tengan más beneficio y porque hemos establecido unos costes por tonelada.

Calculando el margen de beneficio porcentual, con un criterio de selección comprendido entre el 5% y el 10% para evitar riesgos y anomalías en el mercado. Hemos identificado los siguientes países como que cumplen los criteiros.

#### - ¿Cuál es el número de hectáreas para obtener el máximo rendimiento y con qué producto?

In [524]:
# Buscamos la fila exacta que tiene el número más alto en la columna de rendimiento
indice_max = df_agricola['rendimiento_ton_ha'].idxmax()

# Extraemos los datos de esa fila específica
record_absoluto = df_agricola.loc[indice_max, ['pais', 'anio', 'cultivo', 'rendimiento_ton_ha', 'superficie_hectareas', 'produccion_ton']]
print(record_absoluto)

pais                             Rusia
anio                              2018
cultivo                 Caña de azúcar
rendimiento_ton_ha              158.45
superficie_hectareas           3040637
produccion_ton               481787490
Name: 1176, dtype: object


Analizando el conjunto de datos agrícolas en busca del pico histórico de rendimiento. Hemos obtenido que el rendimiento máximo que se ha alcanzado es de 158.45 toneladas por hectárea para lo cual se ha necesitado un total de 3.040.637 hectáreas y se corresponde con el producto de la caña de azúcar. Esto se consiguió en el año 2018 en Rusia.

Pero cabe destacar que el rendimiento es una variable intensiva, lo que significa que la eficiencia productiva de una planta no es una consecuencia matemática directa de disponer de más o menos hectáreas.

Por lo tanto, concluimos que no existe un número mágico de hectáreas que garantice el máximo rendimiento por sí solo; sino que el hito alcanzado con la caña de azúcar demuestra que, con las condiciones climáticas y de inversión en insumos adecuadas, es posible escalar la máxima eficiencia productiva incluso en latifundios de proporciones continentales.

#### - ¿Cuál es el número de cabezas de ganado y tipo para obtener el máximo rendimiento?

In [525]:
df_ganadera['rendimiento_carne_ton_cabeza'] = df_ganadera['produccion_carne_ton'] / df_ganadera['cabezas_ganado']
df_ganadera['rendimiento_leche_lt_cabeza'] = df_ganadera['produccion_leche_lt'] / df_ganadera['cabezas_ganado']

In [526]:
indice_max_carne = df_ganadera['rendimiento_carne_ton_cabeza'].idxmax()
record_carne = df_ganadera.loc[indice_max_carne, ['pais', 'anio', 'tipo_ganado', 'cabezas_ganado', 'rendimiento_carne_ton_cabeza', 'produccion_carne_ton']]

print("----- RÉCORD DE RENDIMIENTO EN CARNE: -----")
print(record_carne)

----- RÉCORD DE RENDIMIENTO EN CARNE: -----
pais                            Alemania
anio                                2011
tipo_ganado                      porcino
cabezas_ganado                    723122
rendimiento_carne_ton_cabeza    2.917004
produccion_carne_ton             2109350
Name: 301, dtype: object


In [528]:
df_lecheros = df_ganadera[df_ganadera['rendimiento_leche_lt_cabeza'] > 0]
if not df_lecheros.empty:
    indice_max_leche = df_lecheros['rendimiento_leche_lt_cabeza'].idxmax()
    record_leche = df_lecheros.loc[indice_max_leche, ['pais', 'anio', 'tipo_ganado', 'cabezas_ganado', 'rendimiento_leche_lt_cabeza', 'produccion_leche_lt']]
    
    print("\n----- RÉCORD DE RENDIMIENTO EN LECHE: -----")
    print(record_leche)


----- RÉCORD DE RENDIMIENTO EN LECHE: -----
pais                              Alemania
anio                                  2013
tipo_ganado                         bovino
cabezas_ganado                      994311
rendimiento_leche_lt_cabeza    9089.002927
produccion_leche_lt             9037295589
Name: 310, dtype: object


El mayor nivel de rendimiento registrado en la producción de carne se localizó en Alemania en el año 2011, siendo el ganado porcino el responsable de ello. El rendmiento alcanzado es de 2.91 toneladas por cabeza, generando un total de más de 2.1 millones de toneladas de carne.

En cuanto al rendimiento en la producción láctea vuelve a ocurrir en Alemania, pero en este caso en el año 2013 con el ganado bovino. Donde se obtuvo un rendimiento de 9.089 litros de leche por cabeza, superando los 9.000 millones de litros producidos.

Pero al igual que pasaba con los cultivos no existe un número mágico de cabezas que dispare el rendimiento. Sino que la eficiencia viene determinada por el tipo de ganado, el tipo de producción que se haga (carne o leche) y las condiciones en las que se tengan a los animales para que sean capaces de producir más y mejor.

#### - ¿Cuáles son los cultivos que mayor resiliencia han demostrado históricamente frente a años de alta escasez hídrica?

In [531]:
# Definimos qué es un año de "Alta Aridez" 
# (Ejemplo: el 25% de los registros con mayor índice de aridez)
df_ag = pd.read_csv('datos/produccion_agricola.csv')
df_clima = pd.read_csv('datos/condiciones_climaticas_limpios.csv')

clima_pais = df_clima.groupby(['anio', 'pais']).mean(numeric_only=True).reset_index()
clima_pais.head()

,anio,pais,dias_con_heladas,humedad_relativa_promedio,indice_aridez,meses_estres_hidrico,precipitacion_total,temperatura_maxima,temperatura_minima,temperatura_promedio
0,2011,Alemania,72.0,76.0,24.860000,0.000000,886.000000,18.200000,1.800000,9.900000
1,2011,Argentina,51.0,69.0,14.753333,4.666667,795.666667,26.133333,4.733333,16.366667
2,2011,Australia,93.6,77.4,14.960000,0.800000,975.800000,31.360000,8.660000,21.000000
3,2011,Brasil,66.0,61.0,8.372000,8.800000,547.000000,31.100000,5.140000,18.620000
4,2011,China,40.0,54.0,13.350000,2.000000,817.000000,25.500000,6.600000,17.000000


In [532]:
df_agricola = pd.merge(df_ag, clima_pais, on=['anio', 'pais'], how='inner')
df_agricola = df_agricola.dropna()

In [533]:

umbral_sequia = df_agricola['indice_aridez'].quantile(0.75)

# Filtramos el dataset en dos escenarios climáticos
df_sequia = df_agricola[df_agricola['indice_aridez'] >= umbral_sequia]
df_normal = df_agricola[df_agricola['indice_aridez'] < umbral_sequia]

# Calculamos el rendimiento medio en ambos escenarios
rendimiento_normal = df_normal.groupby('cultivo')['rendimiento_ton_ha'].mean().reset_index()
rendimiento_sequia = df_sequia.groupby('cultivo')['rendimiento_ton_ha'].mean().reset_index()

# Renombramos columnas para cruzar los datos limpiamente
rendimiento_normal.rename(columns={'rendimiento_ton_ha': 'rendimiento_normal'}, inplace=True)
rendimiento_sequia.rename(columns={'rendimiento_ton_ha': 'rendimiento_sequia'}, inplace=True)

# Cruzamos y calculamos el impacto de la sequía (Variación Porcentual)
df_resiliencia = pd.merge(rendimiento_normal, rendimiento_sequia, on='cultivo', how='inner')

# Fórmula de variación: ((Nuevo - Viejo) / Viejo) * 100
df_resiliencia['impacto_sequia_pct'] = ((df_resiliencia['rendimiento_sequia'] - df_resiliencia['rendimiento_normal']) / df_resiliencia['rendimiento_normal']) * 100

# Ordenamos los resultados. 
# Los cultivos más cercanos a 0% (o positivos) son los más resilientes.
# Los más negativos son los más vulnerables.
top_resilientes = df_resiliencia.sort_values(by='impacto_sequia_pct', ascending=False)

print("\n--- RANKING DE RESILIENCIA (Impacto de la Sequía en el Rendimiento) ---")
print(top_resilientes)


--- RANKING DE RESILIENCIA (Impacto de la Sequía en el Rendimiento) ---
          cultivo  rendimiento_normal  rendimiento_sequia  impacto_sequia_pct
3  Caña de azúcar           86.699706           93.284545            7.594996
2            Café            1.441765            1.496364            3.786952
9              Té            2.035763            2.051429            0.769533
4          Cebada            4.355000            4.334545           -0.469680
7            Soja            3.494151            3.405294           -2.543016
6            Maíz            6.591154            6.321667           -4.088619
5         Girasol            2.512241            2.315455           -7.833118
0         Algodón            2.832778            2.595000           -8.393803
8           Trigo            5.315538            4.846800           -8.818269
1           Arroz            5.847273            5.122667          -12.392206


El análisis que hemos llevado a cabo nos ha llevado a afirmar que la caña de azúcar es el cultivo con mayor resiliencia, seguido del Café y del Té. Al contar con un rendimiento mayor cuando hay mucha aridez demuestra una buena adaptación a los climas cálidos. Mientras que en el extremo opuesto tenemos el arroz cuyo rendimiento se reduce drásticamente cuando hay condiciones extremas con falta de precipitaciones. 

De modo que nos lleva a la conclusión de que para países o regiones con proyecciones de desertificación al alza, se desaconseja rotundamente la dependencia productiva de cultivos altamente vulnerables como el arroz o el trigo.

#### - ¿Qué producto alimentario presenta la mayor volatilidad de precios?

In [534]:
df_precio = pd.read_csv('datos/precios_mercado_limpios.csv')
df_precio.head()

,fecha,pais,producto,precio_usd_ton,volumen_operado_ton,tendencia_30_dias,mercado_principal,precio_min_mes,precio_max_mes,anio_precio,mes_precio,dia_precio
0,2021-01-01,Argentina,Carne_bovina,4299.14,127922,subiendo,UE,3844.74,4517.42,2021,1,1
1,2021-01-01,Argentina,Arroz,330.04,73231,subiendo,Sudeste Asiático,310.10,359.86,2021,1,1
2,2021-01-01,Argentina,Maíz,210.36,11666,bajando,Japón,200.56,238.83,2021,1,1
3,2021-01-01,Brasil,Maíz,212.91,195579,subiendo,Oriente Medio,200.25,243.04,2021,1,1
4,2021-01-01,Brasil,Leche,356.41,136137,subiendo,UE,348.39,390.03,2021,1,1


In [535]:
df_volatilidad = df_precio.groupby('producto')['precio_usd_ton'].agg(
    precio_medio='mean', 
    desviacion_estandar='std'
).reset_index()

# Variación del precio = Riesgo de Mercado
# Fórmula: (Desviación Estándar / Media) * 100
df_volatilidad['volatilidad_riesgo_pct'] = (df_volatilidad['desviacion_estandar'] / df_volatilidad['precio_medio']) * 100

# Ordenamos de mayor a menor riesgo (volatilidad)
df_ranking_riesgo = df_volatilidad.sort_values(by='volatilidad_riesgo_pct', ascending=False)

print("\n--- RANKING DE RIESGO DE MERCADO (Volatilidad de Precios %) ---")
print(df_ranking_riesgo)


--- RANKING DE RIESGO DE MERCADO (Volatilidad de Precios %) ---
        producto  precio_medio  desviacion_estandar  volatilidad_riesgo_pct
4           Maíz    202.287426            23.032484               11.386019
6          Trigo    303.837328            33.135846               10.905785
2  Carne_porcina   2519.861037           221.787755                8.801587
5           Soja    503.806077            44.055913                8.744617
0          Arroz    357.328333            30.935012                8.657307
1   Carne_bovina   4090.084741           315.259573                7.707898
3          Leche    353.613333            25.323889                7.161463


La volatilidad de los precios nos permite evaluar el riesgo en los mercados. Por ello, através del coeficiente de variación de cada producto podemos determinar el riesgo del mercado. De modo, que hemos obtenido que le producto que mayor inestabilidad en los precios es el maíz seguido del trigo, lo que indica que los precios estan expuestos a variables externas. Mientras que la leche y la carne bovina se posicionan como los productos que menor riesgo comercial presentan.
